In [39]:
# Import Libraries
import pandas as pd
from bs4 import BeautifulSoup
import requests
import smtplib
from datetime import time, datetime, timedelta
import random, re
from sklearn.preprocessing import LabelEncoder

# Variable Declaration
global bullet_characters 
global salary_descriptors 
global requirement_descriptors
bullet_characters = ['-', '•', '◦', '▪', '▸', '➤', '★', '*']
salary_descriptors = ['$','pay','paid','salary','compensation']
requirement_descriptors = '(?i)requ|respon|qual|look|skill|know|job|function|Elig|edu|exper|duties|duty'
number = LabelEncoder()

In [2]:
# Extract HTML - Paginated Func
def extract(page):    
    url = f'https://www.teamworkonline.com/jobs-in-sports?employment_opportunity_search%5Bsort_by%5D=most_recent&page={page}'
    user_agents_list = [
    'Mozilla/5.0 (iPad; CPU OS 12_2 like Mac OS X) AppleWebKit/605.1.15 (KHTML, like Gecko) Mobile/15E148',
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/99.0.4844.83 Safari/537.36',
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36',
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/99.0.4844.51 Safari/537.36',
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/144.0.0.0 Safari/537.36"]
    
    headers = {'User-Agent': random.choice(user_agents_list)}

    page = requests.get(url,headers = headers)

    soup = BeautifulSoup(page.content,'html.parser')
    return(soup)

In [3]:
# Extract HTML - Inner Link (a tag) Func
def extract_inner(url):    
    user_agents_list = [
    'Mozilla/5.0 (iPad; CPU OS 12_2 like Mac OS X) AppleWebKit/605.1.15 (KHTML, like Gecko) Mobile/15E148',
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/99.0.4844.83 Safari/537.36',
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36',
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/99.0.4844.51 Safari/537.36',
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/144.0.0.0 Safari/537.36"]
    
    headers = {'User-Agent': random.choice(user_agents_list)}

    page = requests.get(url,headers = headers)

    soup = BeautifulSoup(page.content,'html.parser')
    return(soup)

In [96]:
def transform(jobs_soup):
    all_jobs = jobs_soup.find_all(class_ = "browse-jobs-card")
    raw_job = []
    for job in all_jobs:
##################################################################### Scraping Basic info from Job cards ############################################################################
        try:
            raw_job={
                'logo_image': job.find('img').get('src') if job.find('img') else None,
                'job_url': 'https://www.teamworkonline.com' + job.find('a').get('href') if job.find('a') else None,
                'title': job.find(class_ = "margin-none").get_text().strip() if job.find(class_ = "margin-none") else None,
                'company': job.find(class_ = "browse-jobs-card__content--organization").get_text().strip() if job.find(class_ = "browse-jobs-card__content--organization") else None,
                'location_info': job.find(class_ = "trending__content--small").get_text().strip() if job.find(class_ = "trending__content--small") else None,
                'experience_level': job.find(class_ = "browse-jobs-card__content--small").get_text().strip() if job.find(class_ = "browse-jobs-card__content--small") else None  
            }
####################################################################### Extracting info from the individual job site #################################################################     
            job_soup = extract_inner(raw_job['job_url'])
            raw_job['salary_info'] = salary_fetcher(job_soup)
            raw_job['details'], raw_job['active_job'] = detail_fetcher(job_soup)
                       
################################################### Scraping time based info from Job cards and doing minor extractions  #############################################################    
            posting_number_list = job.find_all(class_ = "browse-jobs-card__scoreboard")
            if len(posting_number_list)>= 2:
                raw_job['posting_numbers'] = posting_number_list[0].get_text().strip() + posting_number_list[1].get_text().strip()
            else:
                raw_job['posting_numbers'] = None
            raw_job['posting_date_identifier'] = job.find(class_ = "trending__content--small trending__scoreboard--time margin-left--xx-small").get_text().strip().split(' ')[0]
            raw_job['scrape_datetime'] = datetime.now()
            
            jobs_list.append(raw_job)
            
        except Exception as e:
            print(f"Error during job scrape: {e}")
            continue
    return

jobs_list = []

In [5]:
# This is code to find salary in the top portion of the sit
def salary_fetcher(job_soup):
    salary_desc = None
    try:
        salary_flag = job_soup.find_all('div',class_ = 'margin-right--x-small')
        for i in range(0,len(salary_flag)):
            if any(descriptor in str(salary_flag[i]) for descriptor in salary_descriptors):
                salary_desc = salary_flag[i].get_text()
            else:
                pass
    except Exception as e:
        print(f"Error during job scrape: {e}")
    return(salary_desc)

In [6]:
####################################### THIS IS THE NEW APPROACH AS OF 3/17 ##############################################
#text = "This is a sample test to see what will happen to be honest: \n\n\n- This is bullet 1.\n\n\n\n- This is bullet 2.\n- This is bullet 3.\n- This is bullet 4.\n- This is bullet 5. \n This is extra text Now that Idk what will happen to."

def wrap_bullets(html_text):
    if(html_text.find('ul')):
        for ul in html_text.find_all('ul'):
            for li in ul:
                li.string = '- ' + li.get_text()
    else:
        pass
    
    text = html_text.get_text(separator = '\n')
    
    lines = text.split('\n')
    result = []
    in_list = False
    
    for line in lines:
        stripped_line = line.strip()
        
        if in_list and stripped_line == '':
            continue
    
        if stripped_line and stripped_line[0] in bullet_characters:
            if not in_list:
                preceding_line = next((line for line in reversed(result) if line.strip()), None)
                result.append(f'<p>{preceding_line}</p>')
                result.append('<ul>')
                in_list = True
            content = stripped_line[1:].strip()
            result.append(f'<li>{content}</li>')
        else:
            if in_list:
                result.append('</ul>')
                in_list = False
            result.append(stripped_line)
            
    if in_list:
        result.append('</ul>')
    
    final_result = '\n'.join(result)
    
    soup = BeautifulSoup(final_result,'html.parser')

    return(soup.find_all('p'))

In [7]:
def detail_fetcher(job_soup):
    try:
        job_inner = job_soup.find('div', class_='opportunity-preview__body')
        active_job_tag = (False if job_soup.find('div', class_ = "opportunity-preview__closed") else True)
        job_details = job_inner.find_all('p')
        
        if job_details:
            if job_inner.find('ul'):
                p_tag = True
            else:
                p_tag = False
        else:
            p_tag = False
              
        if (p_tag):
            pass
        else:
            job_details = wrap_bullets(job_inner)
        
        data = {}
        headers = []
        for section in job_details:
            
            if not section:
                continue
            
            details = []
                
            header_text = section.get_text()
            next_tag = section.find_next_sibling()
            
            if (next_tag and next_tag.name in ['ul', 'ol']):
                headers.append(header_text)
                items = next_tag.find_all('li')
                details = [item.get_text(strip=True) for item in items]
            
                data[header_text] = details 
                    
        
        # Convert to single-row DataFrame
        details_df = pd.DataFrame({k: pd.Series(v) for k, v in data.items()})
    except:
        details_df = []
    return(details_df, active_job_tag)

In [104]:
# Will use this once deployed pages = ((1,3),(3,5),(5,7),(7,9),(9,11))
pages = [(1,3),(3,5)]
for chapter in pages:
    for page in range(chapter[0],chapter[1]):
        recent_jobs_html = extract(page)
        transform(recent_jobs_html)

In [105]:
# Data Cleaning Section - I believe all of this functionally works so far. I need to address hybrid/remote location tags and see if I want to implement them. I also have an issues with repeating data I think so need to inspect that issue
team_work_df = pd.DataFrame(jobs_list)

team_work_df['city'] = team_work_df['location_info'].str.split('·').str[0].str.strip()
team_work_df['state'] = team_work_df['location_info'].str.split('·').str[1].str.strip()
team_work_df.loc[team_work_df['city'].str.contains("[,]"),'city'] = team_work_df['city'].str.split(',').str[0].str.strip()
#team_work_df.loc[team_work_df['location_info'].str.contains('hybrid | remote | united',case = False)]
team_work_df['job_id'] = team_work_df['job_url'].str.split('-').str[-1].str.strip()
team_work_df["company_id"] = number.fit_transform(team_work_df["company"].astype('str'))
team_work_df.loc[team_work_df['company_id'] == 0,'company_id'] = (max(team_work_df['company_id'])+1)
team_work_df.sort_values(by = 'company_id')

team_work_df.loc[~team_work_df['posting_date_identifier'].str.endswith('s'),'posting_date_identifier'] = team_work_df['posting_date_identifier'] + ('s')
team_work_df['posting_source_ID'] = 1
team_work_df['posting_timestamp'] = team_work_df.apply(
    lambda row: row['scrape_datetime'] - timedelta(**{row['posting_date_identifier']: int(row['posting_numbers'])}),
    axis=1)
#team_work_df[team_work_df.duplicated(subset=['job_id'])]
team_work_df[team_work_df['job_id']=='2164610']

,logo_image,job_url,title,company,location_info,experience_level,salary_info,details,active_job,posting_numbers,posting_date_identifier,scrape_datetime,city,state,job_id,company_id,posting_source_ID,posting_timestamp
2,https://cf-production.teamworkonline.com/uploa...,https://www.teamworkonline.com/arenas-faciliti...,Data Analytics & CRM Specialist | Cleveland Br...,Legends Global,Berea · OH,Entry Level,None,ESSENTIAL DUTES AND RESPONSIB...,True,45,minutes,2026-03-31 13:57:38.270756,Berea,OH,2164610,17,1,2026-03-31 13:12:38.270756
100,https://cf-production.teamworkonline.com/uploa...,https://www.teamworkonline.com/arenas-faciliti...,Data Analytics & CRM Specialist | Cleveland Br...,Legends Global,Berea · OH,Entry Level,None,ESSENTIAL DUTES AND RESPONSIB...,True,46,minutes,2026-03-31 13:58:47.795056,Berea,OH,2164610,17,1,2026-03-31 13:12:47.795056
202,https://cf-production.teamworkonline.com/uploa...,https://www.teamworkonline.com/arenas-faciliti...,Data Analytics & CRM Specialist | Cleveland Br...,Legends Global,Berea · OH,Entry Level,None,ESSENTIAL DUTES AND RESPONSIB...,True,01,hours,2026-03-31 14:15:12.656537,Berea,OH,2164610,17,1,2026-03-31 13:15:12.656537


In [10]:
#### THIS IS WHERE IM WORKING FOR THE DETAILS ASPECT AND THEN WILL TACKLE SALARY
## This code takes the nested dataframes in my scraped df and seperates the different requirements back out (I NEED MORE EDITS HERE))
job_requirements_df = pd.concat(
    [row['details'].melt(var_name='header', value_name='detail')
        .dropna(subset=['detail'])
        .assign(job_id=row['job_id'])
     for _, row in team_work_df.iterrows()
     if isinstance(row['details'], pd.DataFrame)],
    ignore_index=True
)

In [11]:
## The code below creates a salary mask to scan the headers/requirements and associated details for any instance of salary information, extracting the numbers from those cells and inserting them into the original salary_info col

salary_mask = job_requirements_df['detail'].str.contains(
    '|'.join([r'\$' if d == '$' else d for d in salary_descriptors]),
    case=False, na=False
) | job_requirements_df['header'].str.contains(
    '|'.join([r'\$' if d == '$' else d for d in salary_descriptors]),
    case=False, na=False
)

salary_results = job_requirements_df[salary_mask]
compensation_mask = salary_results['detail'].str.contains(r'\d', na=False)
compensation_df = salary_results[compensation_mask]

compensation_df['salary_extracted'] = (
    compensation_df['detail'].str.extract(r'(\$[\d,]+(?:\.\d+)?(?:\s?-\s?\$[\d,]+)?(?:\s?(?:K|per|an|\/)\s?\w+)?)')
    .fillna(compensation_df['header'].str.extract(r'(\$[\d,]+(?:\.\d+)?(?:\s?-\s?\$[\d,]+)?(?:\s?(?:K|per|an|\/)\s?\w+)?)'))
)

comp_df = compensation_df.loc[compensation_df['salary_extracted'].notnull(),['job_id','salary_extracted']]

temp_df = team_work_df.merge(
    comp_df,
    on='job_id',
    how='left'
)
team_work_df['salary_info'] = team_work_df['salary_info'].combine_first(temp_df['salary_extracted'])

C:\Users\BigV24\AppData\Local\Temp\ipykernel_18712\2000145957.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  compensation_df['salary_extracted'] = (


In [36]:
## Now that salary is handled we can get rid of details that arent related to job requirements
job_requirements_df.loc[job_requirements_df['header'].str.contains(requirement_descriptors,na=False)].drop('header',axis=1)

,detail,job_id
0,Running player walk-up music,2164528
1,Playing music between innings and during break...,2164528
2,Coordinating with on-field staff to support pr...,2164528
3,"Triggering sound effects (foul balls, crowd hy...",2164528
4,Assisting with overall game day presentation f...,2164528
...,...,...
1019,"Reliable, punctual, and organized",2164451
1020,"Ability to lift 50 lbs (chemicals, equipment)",2164451
1021,Comfortable working outdoors in varying weathe...,2164451
1022,"Ability to stand, bend, and walk for extended ...",2164451


In [46]:

#job_posting_teamwork_df = job_posting_teamwork.reindex(columns = ['job_ID','job_title',"company_ID",'posting_source_ID','posting_datetime','scrape_datetime','application_deadline','salary','job_city','job_state','details','posting_link'])

,logo_image,job_url,title,company,location_info,experience_level,salary_info,details,active_job,posting_numbers,posting_date_identifier,scrape_datetime,city,state,job_id,posting_source_ID,posting_timestamp,company_ID,company_id
85,https://cf-production.teamworkonline.com/uploa...,https://www.teamworkonline.com/baseball-jobs/f...,"Administrative Assistant, Partnerships",Boston Red Sox,Boston · MA,Part Time,NaN,[],True,19,hours,2026-03-31 11:01:27.828782,Boston,MA,2164423,1,2026-03-30 16:01:27.828782,1,1
84,https://cf-production.teamworkonline.com/uploa...,https://www.teamworkonline.com/baseball-jobs/f...,Event Assistant (Part-Time),Boston Red Sox,Boston · MA,Part Time,NaN,[],True,19,hours,2026-03-31 11:01:27.754986,Boston,MA,2164424,1,2026-03-30 16:01:27.754986,1,1
0,https://cf-production.teamworkonline.com/uploa...,https://www.teamworkonline.com/baseball-jobs/n...,Game Day Audio / Music MC,Bristol Blues,Bristol · CT,Intern,NaN,Responsibilities in...,True,34,minutes,2026-03-31 11:01:13.043261,Bristol,CT,2164528,1,2026-03-31 10:27:13.043261,2,2
20,https://cf-production.teamworkonline.com/uploa...,https://www.teamworkonline.com/soccer-jobs/mls...,CCFC Head of Performance,Carolina Core FC,High Point · NC,Manager,NaN,Essential Fu...,True,01,hours,2026-03-31 11:01:17.054976,High Point,NC,2164491,1,2026-03-31 10:01:17.054976,3,3
55,https://cf-production.teamworkonline.com/uploa...,https://www.teamworkonline.com/arenas-faciliti...,Group Sales Intern,Comcast Spectacor,Philadelphia · PA,Intern,NaN,"Responsibilities include, but are not limit...",True,17,hours,2026-03-31 11:01:25.027076,Philadelphia,PA,2164457,1,2026-03-30 18:01:25.027076,4,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29,https://cf-production.teamworkonline.com/uploa...,https://www.teamworkonline.com/hockey-jobs/veg...,Senior Graphic Designer,Vegas Golden Knights,Las Vegas · NV,Manager,NaN,Responsibi...,True,14,hours,2026-03-31 11:01:18.866508,Las Vegas,NV,2164483,1,2026-03-30 21:01:18.866508,38,38
59,https://cf-production.teamworkonline.com/uploa...,https://www.teamworkonline.com/youth-sports-jo...,Volo Sports Weekend Host (Part-time),Volo Sports,Baltimore · MD,Part Time,NaN,[],True,17,hours,2026-03-31 11:01:25.829930,Baltimore,MD,2164449,1,2026-03-30 18:01:25.829930,39,39
57,https://cf-production.teamworkonline.com/uploa...,https://www.teamworkonline.com/youth-sports-jo...,Referee - Soccer,Volo Sports,Baltimore · MD,Part Time,$22.50 - $25 / hour,If you have a desire to make a difference th...,True,17,hours,2026-03-31 11:01:25.577607,Baltimore,MD,2164453,1,2026-03-30 18:01:25.577607,39,39
67,https://cf-production.teamworkonline.com/uploa...,https://www.teamworkonline.com/multiple-proper...,"Assistant, Theater",WME,New York · NY,Entry Level,NaN,[],True,18,hours,2026-03-31 11:01:26.462077,New York,NY,2164442,1,2026-03-30 17:01:26.462077,40,40


logo_image                 11
job_url                    11
title                      11
company                    11
location_info              11
experience_level           11
salary_info                11
details                    11
active_job                 11
posting_numbers            11
posting_date_identifier    11
scrape_datetime            11
city                       11
state                      11
job_id                     11
posting_source_ID          11
posting_timestamp          11
dtype: int64

In [25]:
# Will use this once deployed pages = ((1,3),(3,5),(5,7),(7,9),(9,11))
pages = ((1,3))
for page_range in pages:
    for page_ind in range(page_range[0],page_range[1]):
        recent_jobs_html = extract(page_ind)
        transorm(recent_jobs_html)

1
2
3
4
5
6
7
8
9
10


In [12]:
### This is a copy of scaper code so I can split up some processing

all_jobs = soup.find_all(class_ = "browse-jobs-card")

for job in all_jobs:
##################################################################### Scraping Basic info from Job cards ####################################################
    
    logo_image = job.find('img').get('src')
    job_url = 'https://www.teamworkonline.com' + job.find('a').get('href')
    title = job.find(class_ = "margin-none").get_text().strip()
    company = job.find(class_ = "browse-jobs-card__content--organization").get_text().strip()
    location_info = job.find(class_ = "trending__content--small").get_text().strip()
    # Will need to split as such location_info.split('·')[0])
    experience = job.find(class_ = "browse-jobs-card__content--small").get_text().strip()
    
################################################### Scraping time based info from Job cards and doing minor calcualtions ####################################
    posting_number_list = job.find_all(class_ = "browse-jobs-card__scoreboard")
    posting_numbers = posting_number_list[0].get_text().strip() + posting_number_list[1].get_text().strip()
    posting_date_identifier = job.find(class_ = "trending__content--small trending__scoreboard--time margin-left--xx-small").get_text().strip().split(' ')[0]
    if posting_date_identifier.endswith('s'):
        pass
    else:
        posting_date_identifier = posting_date_identifier + ('s')
    posting_timestamp = datetime.now() - timedelta(**{posting_date_identifier: int(posting_numbers)})

'02/27/2026 12:58:28'

tests


In [20]:
######################### OLD CODE - N0 LONGER USED BUT KEPT BECAUSE ALOT OF WORK ##########################################

#.find_all(['p','strong']) •
data = {}
headers = []
for section in job_details:
    if (p_tag):
        full_text = section
        
        if section.find('strong'):
            header = section.find('strong').get_text()
            content = section.find_next_sibling()
        else:
            header = ''
            content = ''
    else:
        full_text = wrap_bullets(section.get_text(separator = '\n'))
        print(type(full_text))
    #for br in section.find_all('br'):
    #    br.replace_with("\n")
    
    #full_text = section.get_text(separator = '\n')
    if not full_text:
        continue
    
    #print(full_text)
    #header_text,details = extract_bullets_from_text(full_text,header,content)
    
    #if header_text and details:
    #    headers.append(header_text)
    #    data[header_text] = details
    #else:
    details = []
        
    header_text = full_text.get_text()
    next_tag = full_text.find_next_sibling()
    
    if (next_tag and next_tag.name in ['ul', 'ol']):
        headers.append(header_text)
        items = next_tag.find_all('li')
        details = [item.get_text(strip=True) for item in items]
    
        data[header_text] = details 
            

# Convert to single-row DataFrame
df = pd.DataFrame({k: pd.Series(v) for k, v in data.items()})
print(df)


def extract_bullets_from_text(text,strong_header,strong_content):
    """Extract header and bullets from text"""
    try:
        if strong_content and strong_header:
            header = strong_header
            content = strong_content.get_text().strip()
        else:
            if ':' in text:
                sections = text.split(':')
                #print(sections)
                header = sections[0].strip()
                content = sections[1:]
            elif '\n' in text:
                sections = text.split('\n', 1)
                header = sections[0].strip()
                content = sections[1].strip() if sections[1].strip() != '' else header
                
                
        if header and content:
            bullets = re.split(r'[-•·*]\s*', content)
            
            if len(bullets) <= 1:
                bullets = re.split(r'-\s+', content)
            
            bullets = [b.strip() for b in bullets if b.strip()]
            
            if bullets:
                return (header, bullets)
        
    except:
        print('Error')
        pass
    
    return (None, None)

AttributeError: 'str' object has no attribute 'get_text'

In [33]:
job_soup = extract_inner('https://www.teamworkonline.com/hockey-jobs/hockeyjobs/nhl-league-office/royalty-analyst-2163333')

In [8]:
################## FOR NOW I HAVE THE DATA I BELEIVE I NEED. TIME TO TAKE TIME TO LAY OUT THE DATABASE SCHEMA #############

In [80]:
#temp_df['salary_info'].loc[temp_df['title'] == 'Senior Manager, Commercial Strategy & Operations']

In [9]:
active_job_tag

NameError: name 'active_job_tag' is not defined

In [181]:
headers

['The royalty analyst will be responsible for the day-to-day processing of consumer products royalty statements. Tasks include:',
 'Required',
 'Preferred',
 'Education/Certifications',
 'Required Skills',
 'These core competencies reflect the underlying values that are necessary to represent the National Hockey League:']

In [182]:
# CODE TO PULL OFF RELEVANT INFO FROM DETAIL SCRAPE
df.filter(regex = '(?i)requ|respon|qual|look|skill|know|job|function|Elig|edu|exper|duties|duty',axis=1)

,The royalty analyst will be responsible for the day-to-day processing of consumer products royalty statements. Tasks include:,Required,Education/Certifications,Required Skills
0,Processing statements in the royalty system,Advanced Microsoft Excel skills,"Minimum of Bachelor’s degree, Accounting or Fi...",Excellent verbal and written skills
1,Reviewing royalties reported for assigned lice...,NaN,NaN,Strong analytical and organizational skills an...
2,Analyzing revenue generated from royalty state...,NaN,NaN,Able to handle multiple tasks simultaneously
3,Reviewing and applying cash received from lice...,NaN,NaN,Take initiative and develop creative solutions...
4,Communicate with licensees regarding their roy...,NaN,NaN,NaN
5,Assisting with the preparation of royalty repo...,NaN,NaN,NaN
6,Assist with testing new releases of the royalt...,NaN,NaN,NaN
7,Ad hoc special projects,NaN,NaN,NaN


In [183]:
details_df = []
for header in headers:
    details_df.append({header: df[header].str.split('\n').explode(header)})
details_df = pd.DataFrame(details_df)
details_df

,The royalty analyst will be responsible for the day-to-day processing of consumer products royalty statements. Tasks include:,Required,Preferred,Education/Certifications,Required Skills,These core competencies reflect the underlying values that are necessary to represent the National Hockey League:
0,0 Processing statements in the royalt...,NaN,NaN,NaN,NaN,NaN
1,NaN,0 Advanced Microsoft Excel skills 1 ...,NaN,NaN,NaN,NaN
2,NaN,NaN,0 Experience with consumer products licensi...,NaN,NaN,NaN
3,NaN,NaN,NaN,"0 Minimum of Bachelor’s degree, Accounting ...",NaN,NaN
4,NaN,NaN,NaN,NaN,0 Excellent verbal and writte...,NaN
5,NaN,NaN,NaN,NaN,NaN,0 Accountability 1 ...
